# Patrimonio protegido de Madrid


Distribución, concentración y cambios en la protección arquitectónica de la ciudad.





**Carga de liberias y datos.**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sbs
import matplotlib.pyplot as plt
import folium
from pyproj import Transformer
from folium.plugins import MarkerCluster, HeatMap

plt.style.use('ggplot')

df = pd.read_csv('CATALOGO_EDIFICIOS_PROTEGIDOS.csv', sep=';')

**Prepación de datos.**

In [ ]:
df.rename(columns={
    'PROTECCION_ACTUAL': 'proteccion_actual',
    'PROTECCION_97': 'proteccion_1997',
    'UTMX_ETRS': 'utmx',
    'UTMY_ETRS': 'utmy'
}, inplace=True)

**Conversión geoespacial.**

In [ ]:
transformer = Transformer.from_crs("epsg:25830", "epsg:4326", always_xy=True)

lon, lat = transformer.transform(df['utmx'].values, df['utmy'].values)

df['lon'] = lon
df['lat'] = lat

**KPIs principales**

In [ ]:
total_edificios = len(df)
niveles_proteccion = df['proteccion_actual'].nunique()

print("Edificios protegidos:", total_edificios)
print("Niveles de protección:", niveles_proteccion)

Distribución por nivel de protección

In [ ]:
plt.figure(figsize=(10,6))


order = df['proteccion_actual'].value_counts().index

ax = sbs.countplot(
    data=df,
    y='proteccion_actual',
    order=order,
    palette='Spectral'
)


plt.title('Distribución de niveles de protección', fontsize=16, weight='bold')


plt.xlabel('')
plt.ylabel('')


ax.set_facecolor('none')
plt.gcf().patch.set_alpha(0)

sbs.despine(left=True, bottom=True)


for p in ax.patches:
    width = p.get_width()
    ax.text(
        width + 5,
        p.get_y() + p.get_height() / 2,
        int(width),
        va='center',
        fontsize=11
    )

plt.show()

Evolución histórica · Cambios desde 1997

In [ ]:
pd.crosstab(df['proteccion_actual'], df['proteccion_1997'])

# **Mapa principal**

In [ ]:
mapa = folium.Map(
    location=[df['lat'].mean(), df['lon'].mean()],
    zoom_start=12,
    tiles='cartodbpositron'
)

cluster = MarkerCluster().add_to(mapa)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        fill=True
    ).add_to(cluster)

mapa

# **Heatmap**

In [ ]:
heat_data = df[['lat', 'lon']].values.tolist()

HeatMap(heat_data, radius=8).add_to(mapa)

mapa